In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import lightgbm as lgb
import time


from data_loader import split_data
from data_loader import test_data_func

from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder



In [ ]:
path = r'data/train.csv'
setting = 'df'

lst_1 = ['RoofMatl', 'HouseStyle', 'LandContour', 'LandSlope', 'Alley', 'BldgType', 'Street', 'MSZoning', 'RoofStyle', 'LowQualFinSF', 'Heating', 'BsmtFinType2', 'Utilities', 'Condition2', 'HeatingQC', 'Electrical', 'PoolArea']

df = split_data(path, setting, lst_1)
df.head()

### Encoded cat and int64 cols with OHE and Standard Scalar respectively

- Dataset now has 1400x300 shape and ready for Ridge and Lasso modeling

In [ ]:
df_ = df.copy()

cat_cols = df_.select_dtypes(include=['string']).columns
for col in cat_cols:
    df_[col] = df_[col].astype('category')


ind_int = df_.dtypes[df.dtypes == 'int64'].index[:-1]
cat_cols = df_.dtypes[df_.dtypes == 'category'].index

scaler = StandardScaler()

df_[ind_int] = scaler.fit_transform(df_[ind_int])

one_hot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_data = one_hot.fit_transform(df[cat_cols])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=one_hot.get_feature_names_out(cat_cols),
    index=df_.index
)

final_df = pd.concat(
    [df_.drop(columns=cat_cols), encoded_df],
    axis=1
)


final_df = final_df.fillna(0)
final_df['SalePrice'] = np.log1p(final_df['SalePrice'])

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split


y = final_df['SalePrice']
X = final_df.drop(['SalePrice', 'Id'], axis=1)

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size=0.25, shuffle=True, random_state=42)

lasso_cv = LassoCV(eps=0.1,alphas=10, cv=5,random_state=42)

lasso_cv.fit(X_train, y_train)

preds = lasso_cv.predict(X_val)

print(f'MSE: {mean_squared_error(y_val, preds)}')



In [ ]:
fig, ax = plt.subplots(figsize=(7,4))

ls = [np.mean(x) for x in lasso_cv.mse_path_]
ax.plot(ls)

In [ ]:
for x in range(len(y_val)):
    print(f'VAL {np.expm1(y_val.to_list()[x])} : PRED: {np.expm1(preds[x])}')